# Train the filament U-Nets (ImageNet-pretrained encoders) on a Kaggle GPU
Settings (right panel): **Accelerator → GPU T4 x2** and **Internet → On** (needed to download the pretrained encoder).
Inputs: the competition data and your `filseg-code` dataset.
Run all cells (**Run All**). Training is limited to `MINUTES`. When it finishes, download `unet_results.zip` from **Output** (right panel).

This is **Version 4** of the Kaggle notebook (run 11 Nov 2026, 4 h 48 min). It trains two models at the same time, one per T4 GPU:
* GPU 1: **U-Net + ResNet-50 encoder**, 512 px crops, batch 8, 240 min → `weights/unet_r50.pt` (**final model**, best validation Dice 0.707).
* GPU 0: U-Net + ResNet-34 encoder, 768 px crops, batch 8, 240 min → `weights/unet.pt` here (a trial; worse than the 512 px version, not used).

The earlier ResNet-34 model (`weights/unet_r34.pt` in the repo) came from **Version 3**: the same notebook with `MINUTES = 150` and one
training command, `!python scripts/run.py train --data /kaggle/working {ARGS} {MODEL} --minutes {MINUTES}`.

In [ ]:
import glob, os, shutil, subprocess, sys
# find the competition data and the uploaded code, wherever Kaggle mounted them
JSON = glob.glob("/kaggle/input/**/MAGFiLO_1.0_Annotations_kaggle2026_train.json", recursive=True)[0]
ROOT = os.path.dirname(os.path.dirname(JSON))                     # .../MAGFiLO_1.0_Kaggle_2026
TRAIN_DIR = os.path.join(ROOT, "train", "train_images")
TEST_DIR  = os.path.join(ROOT, "test", "test_images")
RUN = glob.glob("/kaggle/input/**/scripts/run.py", recursive=True)[0]
CODE = os.path.dirname(os.path.dirname(RUN))
shutil.copytree(CODE, "/kaggle/working/filseg", dirs_exist_ok=True,
                ignore=shutil.ignore_patterns("data", "*.zip", "__pycache__"))
os.chdir("/kaggle/working/filseg")
os.makedirs("weights", exist_ok=True)
print(JSON, TRAIN_DIR, TEST_DIR, os.getcwd(), sep="\n")
import torch; print("GPU:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

In [ ]:
!pip install -q segmentation-models-pytorch==0.5.0

In [ ]:
MINUTES = 240   # training time budget; Kaggle sessions can run up to 12 h (Version 3 used 150)
ARGS = f'--train-json "{JSON}" --train-dir "{TRAIN_DIR}"'
# ImageNet-pretrained ResNet-34 encoder, larger crops and batch on the GPU (used by Version 3)
MODEL = "--arch smp --encoder resnet34 --crop 512 --batch 16 --lr 5e-4"

## 1. Train (Version 4: two models at once)
The input cache is built once first, because two training runs writing the same cache files at the same time could clash.

In [ ]:
# Version 4: build the input cache ONCE, then train two models at the same time (one per T4 GPU)
sys.path[:0] = ["src", "scripts"]
from filseg import CocoSet, unet
from run import split_ids
coco = CocoSet(JSON, TRAIN_DIR)
tr, va = split_ids(coco)
unet.build_cache(coco, tr, "/kaggle/working/_cache1024")
unet.build_cache(coco, va[:20], "/kaggle/working/_cache1024")
jobs = [("0", "resnet34", 768, 8, "unet"),        # GPU 0: same encoder, larger 768 px crops
        ("1", "resnet50", 512, 8, "unet_r50")]    # GPU 1: bigger ResNet-50 encoder
procs = []
for gpu, enc, crop, batch, name in jobs:
    cmd = (f"python scripts/run.py train --data /kaggle/working {ARGS} --arch smp --encoder {enc} "
           f"--crop {crop} --batch {batch} --lr 5e-4 --minutes {MINUTES} --weights weights/{name}.pt "
           f"2>&1 | tee weights/{name}.log")
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=gpu, PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True")
    procs.append(subprocess.Popen(cmd, shell=True, env=env))
print("exit codes:", [p.wait() for p in procs])

## 2. Tune threshold / fragment linking for PQ on the validation split
This cell tunes `weights/unet.pt` only (in Version 4: the 768 px ResNet-34 trial). The final ResNet-50 was tuned and
scored on the laptop on all 106 validation files: `scripts/cache_probs.py` + `scripts/postproc_lab.py combo`
(result in `weights/postprocess_unet_r50.json`).

In [ ]:
!python scripts/run.py tune --method unet {ARGS} --n-val 60

## 3. Submission file

In [ ]:
!python scripts/run.py submit --method unet --test-dir "{TEST_DIR}" --out /kaggle/working/submission.csv

## 4. Files to download (Output panel → /kaggle/working/filseg/weights) and copy into `Documents\\filament-seg\\weights`

In [ ]:
shutil.copy("/kaggle/working/submission.csv", "/kaggle/working/filseg/weights/submission_unet.csv")
for f in sorted(os.listdir("weights")): print(f, os.path.getsize("weights/" + f) // 1024, "KB")
shutil.make_archive("/kaggle/working/unet_results", "zip", "weights")   # one zip with everything
print("Download /kaggle/working/unet_results.zip")